# 11 — Code Execution (Vertex AI)

**AWS-side equivalent:** AgentCore CodeInterpreter (Python sandbox)
**GCP service:** Vertex AI Code Execution tool, exposed as a built-in tool on Gemini

AWS-side notebook: `notebooks/aws/11-code-execution.ipynb`

_As of 2026-05: the supported shape is `types.Tool(code_execution=types.ToolCodeExecution())` passed via `config=` on the unified `google.genai` client. The legacy `vertexai.generative_models.Tool.from_code_execution()` still works but is deprecated. `gemini-2.5-pro` and `gemini-2.5-flash` both support the tool; the canonical model for analysis tasks is `gemini-2.5-pro`._


## Steps

1. Build a `google.genai` client targeting Vertex AI in the project's region
2. Invoke Gemini with the `code_execution` tool on a `import statistics; print(...)` prompt
3. Walk the response parts to extract the executable code, stdout, and the model's narration
4. Mirror the AWS notebook's surface: capture stdout, captured code, and a JSON result payload


## Step 1: Load config


In [ ]:
import json
from pathlib import Path

config_path = Path('config.json')
config = json.loads(config_path.read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
MODEL_ID = gcp.get('model_id', 'gemini-2.5-pro')
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'
print(f'project={PROJECT_ID} region={REGION} model={MODEL_ID}')


## Step 2: Initialise the `google.genai` client (Vertex AI mode)

The new `google-genai` SDK is provider-aware: passing `vertexai=True` routes calls through Vertex AI in the project + region you supply (instead of the public Gemini API endpoint). ADC handles auth — no API key needed inside the project.


In [ ]:
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)
print(f'✓ genai client ready (vertexai mode, project={PROJECT_ID}, location={REGION})')


## Step 3: Invoke Gemini with the code-execution tool

We ask Gemini to compute summary stats over a small array using the `statistics` module — a deliberately simple snippet that exercises stdout capture without depending on `pandas` or `numpy`.


In [ ]:
PROMPT = (
    'You are OrgIQ Analyst. Using a Python code-execution block, compute the mean, median, '
    'standard deviation, and min/max of the OrgIQ expertise scores '
    '[0.45, 0.72, 0.88, 0.91, 0.60, 0.55, 0.78, 0.83, 0.50, 0.95]. '
    'Use the standard-library `statistics` module. Print the result as a JSON object so it is '
    'easy to parse, then give a one-sentence interpretation.'
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution())],
        temperature=0.0,
    ),
)
print(f'✓ response received ({len(response.candidates)} candidates)')


## Step 4: Walk the response parts

A code-execution response interleaves three kinds of parts:

| Part attribute | Meaning |
|---|---|
| `part.text` | Narrative the model emitted around the code |
| `part.executable_code` | The Python the model asked the sandbox to run (`.code`, `.language`) |
| `part.code_execution_result` | The captured output (`.output`, `.outcome`) |

We unpack all three and surface them the same way the AgentCore CodeInterpreter notebook surfaces `stdout`, `stderr`, and the generated source.


In [ ]:
narration: list[str] = []
executed_code: list[str] = []
stdout_chunks: list[str] = []
outcomes: list[str] = []

for cand in response.candidates:
    for part in cand.content.parts:
        if getattr(part, 'text', None):
            narration.append(part.text)
        if getattr(part, 'executable_code', None):
            ec = part.executable_code
            executed_code.append(ec.code)
            print(f'--- executable_code ({getattr(ec, "language", "PYTHON")}) ---')
            print(ec.code)
        if getattr(part, 'code_execution_result', None):
            cer = part.code_execution_result
            stdout_chunks.append(cer.output or '')
            outcomes.append(str(getattr(cer, 'outcome', 'OK')))
            print(f'--- code_execution_result (outcome={outcomes[-1]}) ---')
            print(cer.output)

print(f'\n✓ parsed {len(executed_code)} code block(s), {len(stdout_chunks)} stdout chunk(s)')


## Step 5: Parse the JSON result and show the narration

Because we asked for JSON output, we can pull a structured payload out of stdout — the same downstream contract the AWS notebook relies on for charting / persisting results.


In [ ]:
import re

stats_payload = None
for chunk in stdout_chunks:
    m = re.search(r'\{.*\}', chunk, re.DOTALL)
    if m:
        try:
            stats_payload = json.loads(m.group(0))
            break
        except json.JSONDecodeError:
            continue

print('Narration:')
for n in narration:
    print(f'  {n.strip()}')

print('\nParsed stats:')
print(json.dumps(stats_payload, indent=2) if stats_payload else '  (no JSON found in stdout)')


## Step 6: Persist the result to `config.json`

Tracks the model id used so notebook 14 (scoring) and 16 (eval) can pin to the same canonical model.


In [ ]:
gcp['code_execution'] = {
    'model_id': MODEL_ID,
    'last_outcome': outcomes[-1] if outcomes else None,
    'last_stats': stats_payload,
    'last_code_chars': sum(len(c) for c in executed_code),
}
config_path.write_text(json.dumps(config, indent=2))
print('✓ config.json updated under gcp.code_execution')


## Notes & caveats

- Code execution shares the model's safety filters; outputs may be blocked even if the snippet is harmless. Inspect `cand.finish_reason` if `executable_code` is empty.
- The sandbox has no outbound network and only a curated stdlib + `numpy`/`pandas`/`matplotlib` surface. Treat it the same as the AgentCore CodeInterpreter.
- If you need the legacy SDK for parity with older notebooks: `from vertexai.generative_models import GenerativeModel, Tool; tool = Tool.from_code_execution()`. This path emits a `DeprecationWarning` as of the 1.71 release of the `vertexai` SDK; prefer `google.genai`.


## Done

Next: `12-ag-ui-streaming.ipynb` (wire this same model up to an SSE stream of AG-UI events).
